# Qwen3 embedding scale comparison: 0.6B vs 4B vs 8B

Very preliminary exploratory analysis of the same SCORE concept table embedded with three Qwen3-Embedding models.

Questions:

1. Does scaling improve concept → definition matching?
2. Are definitions and their paraphrases stable in all three models?
3. Do the three models induce broadly similar pairwise geometry?
4. How do ordinary English semantic relations behave in the same embedding models?
5. Do selected difficult EEG contrasts become better separated as model size increases?

Two spaces are available for every model:

- **native**: 1024 / 2560 / 4096 dimensions;
- **1024**: first 1024 MRL dimensions, re-normalized, for a dimension-matched comparison.

The 1024-D spaces do **not** share aligned coordinate axes. Cross-model comparison therefore uses similarities, ranks and similarity-matrix agreement rather than direct subtraction of vectors from different models.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from sklearn.decomposition import PCA

OUT = Path("embedding_results")

MODEL_FILES = {
    "0.6B": "0_6b",
    "4B": "4b",
    "8B": "8b",
}

score_meta = pd.read_csv(OUT / "score_embeddings_metadata.csv")
natural_meta = pd.read_csv(OUT / "natural_language_embeddings_metadata.csv")
external_meta = pd.read_csv(OUT / "external_domain_embeddings_metadata.csv")

score = {}
natural = {}
external = {}

for model, slug in MODEL_FILES.items():
    s = np.load(OUT / f"score_qwen3_{slug}.npz")
    n = np.load(OUT / f"natural_qwen3_{slug}.npz")
    e = np.load(OUT / f"external_qwen3_{slug}.npz")

    score[model] = {
        "native": s["embeddings_native"],
        "1024": s["embeddings_1024"],
    }
    natural[model] = {
        "native": n["embeddings_native"],
        "1024": n["embeddings_1024"],
    }
    external[model] = {
        "native": e["embeddings_native"],
        "1024": e["embeddings_1024"],
    }

print("SCORE metadata:", score_meta.shape)
print("Natural-language metadata:", natural_meta.shape)
print("External-domain metadata:", external_meta.shape)
print(external_meta["domain"].value_counts())

for model in MODEL_FILES:
    print(
        model,
        "SCORE native", score[model]["native"].shape,
        "SCORE 1024", score[model]["1024"].shape,
        "natural native", natural[model]["native"].shape,
    )

## 1. Basic validation

Every SCORE row should contribute exactly 9 embeddings: concept name, definition and 7 paraphrases.

In [ ]:
print("number of SCORE concepts:", score_meta["row_id"].nunique())
print(score_meta.groupby("row_id").size().value_counts())
print()
print(score_meta["type"].value_counts())

for model in MODEL_FILES:
    for space in ["native", "1024"]:
        X = score[model][space]
        norms = np.linalg.norm(X, axis=1)
        print(
            model, space,
            "norm range:",
            float(norms.min()),
            float(norms.max())
        )

## 2. Concept name → own definition

This is deliberately difficult because a short label and a full definition have very different surface form.

In [ ]:
def concept_definition_metrics(X, meta):
    c_mask = meta["type"].eq("concept").to_numpy()
    d_mask = meta["type"].eq("definition").to_numpy()

    C = X[c_mask]
    D = X[d_mask]

    c_rows = meta.loc[c_mask, "row_id"].to_numpy()
    d_rows = meta.loc[d_mask, "row_id"].to_numpy()
    assert np.array_equal(c_rows, d_rows)

    S = C @ D.T
    own = np.diag(S)

    off_diag = ~np.eye(len(S), dtype=bool)
    other = S[off_diag]

    ranks = 1 + (S > own[:, None]).sum(axis=1)

    return {
        "own_mean": own.mean(),
        "other_mean": other.mean(),
        "separation": own.mean() - other.mean(),
        "top1": np.mean(ranks == 1),
        "top5": np.mean(ranks <= 5),
        "median_rank": np.median(ranks),
    }


rows = []

for model in MODEL_FILES:
    for space in ["1024", "native"]:
        rows.append({
            "model": model,
            "space": space,
            **concept_definition_metrics(
                score[model][space],
                score_meta
            )
        })

concept_definition_table = pd.DataFrame(rows)
display(concept_definition_table.round(4))

The **1024** rows are the cleanest model-size comparison because vector dimensionality is matched. The **native** rows show each model at full embedding dimensionality.

## 3. Definition → own paraphrases and paraphrase coherence

This is a positive control: all seven paraphrases were intended to preserve the original definition.

In [ ]:
def paraphrase_metrics(X, meta):
    def_indices = meta.index[meta["type"].eq("definition")].to_numpy()

    own_def_para = []
    within_para = []

    for d_idx in def_indices:
        row_id = meta.loc[d_idx, "row_id"]

        p_idx = meta.index[
            meta["row_id"].eq(row_id) &
            meta["type"].eq("paraphrase")
        ].to_numpy()

        d = X[d_idx]
        P = X[p_idx]

        own_def_para.extend(P @ d)

        S = P @ P.T
        upper = np.triu_indices(len(P), k=1)
        within_para.extend(S[upper])

    return {
        "definition_paraphrase_mean": np.mean(own_def_para),
        "definition_paraphrase_min": np.min(own_def_para),
        "paraphrase_pair_mean": np.mean(within_para),
        "paraphrase_pair_min": np.min(within_para),
    }


rows = []

for model in MODEL_FILES:
    for space in ["1024", "native"]:
        rows.append({
            "model": model,
            "space": space,
            **paraphrase_metrics(score[model][space], score_meta)
        })

display(pd.DataFrame(rows).round(4))

## 4. Geometry of concept centroids

For each concept, average its definition and seven paraphrases, then L2-normalize the centroid. This reduces direct influence of the very short concept label.

In [ ]:
def concept_centroids(X, meta):
    centroids = []
    names = []
    row_ids = []

    for row_id, g in meta.groupby("row_id", sort=True):
        idx = g.index[
            g["type"].isin(["definition", "paraphrase"])
        ].to_numpy()

        c = X[idx].mean(axis=0)
        c = c / np.linalg.norm(c)

        centroids.append(c)
        names.append(g["concept"].iloc[0])
        row_ids.append(row_id)

    return np.vstack(centroids), pd.DataFrame({
        "row_id": row_ids,
        "concept": names
    })


centroids = {}
centroid_meta = None

for model in MODEL_FILES:
    centroids[model] = {}

    for space in ["1024", "native"]:
        C, cm = concept_centroids(
            score[model][space],
            score_meta
        )

        centroids[model][space] = C

        if centroid_meta is None:
            centroid_meta = cm

display(centroid_meta.head())

### Agreement of global pairwise geometry

Spearman correlation is computed between all upper-triangle centroid similarities. High values mean that models broadly agree about which concept pairs are relatively closer or farther apart.

In [ ]:
def upper_triangle_cosines(X):
    S = X @ X.T
    mask = np.triu(np.ones_like(S, dtype=bool), k=1)
    return S[mask]


for space in ["1024", "native"]:
    print("\nSPACE:", space)

    vals = {
        model: upper_triangle_cosines(
            centroids[model][space]
        )
        for model in MODEL_FILES
    }

    corr = pd.DataFrame(
        index=MODEL_FILES,
        columns=MODEL_FILES,
        dtype=float
    )

    for a in MODEL_FILES:
        for b in MODEL_FILES:
            corr.loc[a, b] = spearmanr(
                vals[a],
                vals[b]
            ).statistic

    display(corr.round(3))

## 5. Selected difficult EEG contrasts

Pairs are included only if their names exist in the final CSV. Lower cosine is not automatically "better"; the question is whether scaling changes separation consistently.

In [ ]:
candidate_pairs = [
    ("Rhytmic trains/bursts", "Arhythmic trains/bursts"),
    ("Rhythmic trains/bursts", "Arhythmic trains/bursts"),

    ("Background delta activity", "Background theta activity"),
    ("Excessive interictal delta activity", "Excessive interictal theta activity"),
    ("Excessive delta activity", "Excessive theta activity"),
    ("Physiologic delta activity", "Physiologic theta activity"),

    ("Atonic seizure", "Atonic motor manifestation"),
    ("Clonic seizure", "Clonic motor manifestation"),
    ("Myoclonic seizure", "Myoclonic motor manifestation"),
    ("Tonic seizure", "Tonic motor manifestation"),
]

available = set(centroid_meta["concept"])
pairs = []
seen = set()

for a, b in candidate_pairs:
    if a in available and b in available and (a, b) not in seen:
        pairs.append((a, b))
        seen.add((a, b))


def centroid_index(name):
    return centroid_meta.index[
        centroid_meta["concept"].eq(name)
    ][0]


rows = []

for a, b in pairs:
    ia, ib = centroid_index(a), centroid_index(b)
    row = {"concept_1": a, "concept_2": b}

    for model in MODEL_FILES:
        C = centroids[model]["1024"]
        row[model] = float(C[ia] @ C[ib])

    rows.append(row)

hard_pairs = pd.DataFrame(rows)

if len(hard_pairs):
    display(hard_pairs.round(4))
else:
    print("No candidate hard-negative pairs matched the final concept names.")

## 6. Natural-language control words

These are diagnostics, not a formal benchmark. Antonyms can be close in embedding space because they often occur in similar contexts.

In [ ]:
word_to_idx = {
    word: i
    for i, word in enumerate(natural_meta["word"])
}

def word_vector(X, word):
    return X[word_to_idx[word]]

pairs = [
    ("black", "white"),
    ("king", "queen"),
    ("man", "woman"),
    ("young", "old"),
    ("mother", "father"),
    ("small", "big"),
    ("dark", "light"),
    ("open", "closed"),
    ("regular", "irregular"),
    ("patience", "impatience"),
    ("partial", "impartial"),
    ("typical", "atypical"),
]

rows = []

for a, b in pairs:
    row = {"word_1": a, "word_2": b}

    for model in MODEL_FILES:
        X = natural[model]["1024"]
        row[model] = float(
            word_vector(X, a) @ word_vector(X, b)
        )

    rows.append(row)

display(pd.DataFrame(rows).round(4))

### Relation-vector alignment within each model

In [ ]:
def unit(x):
    return x / np.linalg.norm(x)

relation_groups = {
    "gender": [
        ("man", "woman"),
        ("king", "queen"),
        ("father", "mother"),
    ],
    "morphological_opposition": [
        ("regular", "irregular"),
        ("patience", "impatience"),
        ("partial", "impartial"),
        ("typical", "atypical"),
    ],
}

for group_name, rels in relation_groups.items():
    print("\nRELATION GROUP:", group_name)

    summary = []

    for model in MODEL_FILES:
        X = natural[model]["1024"]

        names = [f"{a}→{b}" for a, b in rels]
        R = np.vstack([
            unit(
                word_vector(X, b) -
                word_vector(X, a)
            )
            for a, b in rels
        ])

        S = R @ R.T
        mask = np.triu(
            np.ones_like(S, dtype=bool),
            k=1
        )

        summary.append({
            "model": model,
            "mean_relation_alignment": S[mask].mean(),
            "min_relation_alignment": S[mask].min(),
        })

        print("\n", model)
        display(
            pd.DataFrame(
                S,
                index=names,
                columns=names
            ).round(3)
        )

    print("Summary:")
    display(pd.DataFrame(summary).round(3))

### Ordered age sequence

In [ ]:
rows = []

for model in MODEL_FILES:
    X = natural[model]["1024"]

    d1 = unit(
        word_vector(X, "child") -
        word_vector(X, "baby")
    )
    d2 = unit(
        word_vector(X, "adult") -
        word_vector(X, "child")
    )

    rows.append({
        "model": model,
        "cosine(baby→child, child→adult)": float(d1 @ d2)
    })

display(pd.DataFrame(rows).round(4))

### Tiny-vocabulary analogies

The vocabulary is intentionally tiny, so treat this only as a sanity check.

In [ ]:
def analogy(X, a, b, c, n=5):
    target = unit(
        word_vector(X, b) -
        word_vector(X, a) +
        word_vector(X, c)
    )

    sims = X @ target

    for w in [a, b, c]:
        sims[word_to_idx[w]] = -np.inf

    order = np.argsort(sims)[::-1][:n]

    return pd.DataFrame({
        "word": natural_meta.iloc[order]["word"].to_numpy(),
        "cosine": sims[order],
    })


for model in MODEL_FILES:
    print("\nMODEL:", model)
    X = natural[model]["1024"]

    print("man : woman :: king : ?")
    display(analogy(X, "man", "woman", "king"))

    print("man : woman :: father : ?")
    display(analogy(X, "man", "woman", "father"))

    print("regular : irregular :: typical : ?")
    display(analogy(X, "regular", "irregular", "typical"))

## 7. PCA of ordinary-language controls

PCA axes are fitted separately for every model. Do not compare PC1/PC2 directions between models.

In [ ]:
for model in MODEL_FILES:
    X = natural[model]["1024"]

    pca = PCA(n_components=2)
    Z = pca.fit_transform(X)

    plt.figure(figsize=(9, 7))
    plt.scatter(Z[:, 0], Z[:, 1])

    for i, word in enumerate(natural_meta["word"]):
        plt.annotate(
            word,
            (Z[i, 0], Z[i, 1]),
            fontsize=9
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(
        f"{model} — natural-language controls\n"
        f"variance explained = {pca.explained_variance_ratio_.sum():.3f}"
    )
    plt.show()

## 8. Cross-domain regularity direction

If present in SCORE, compare ordinary English `regular → irregular` with EEG `Rhythmic/Rhytmic trains/bursts → Arhythmic trains/bursts` within each model.

In [ ]:
def score_concept_name_vector(X, name):
    idx = score_meta.index[
        score_meta["type"].eq("concept") &
        score_meta["concept"].eq(name)
    ].to_numpy()

    if len(idx) != 1:
        return None

    return X[idx[0]]


rhythmic_candidates = [
    "Rhythmic trains/bursts",
    "Rhytmic trains/bursts",
]

rhythmic_name = next(
    (
        x for x in rhythmic_candidates
        if x in set(score_meta["concept"])
    ),
    None
)

arhythmic_name = (
    "Arhythmic trains/bursts"
    if "Arhythmic trains/bursts" in set(score_meta["concept"])
    else None
)

if rhythmic_name and arhythmic_name:
    rows = []

    for model in MODEL_FILES:
        NW = natural[model]["1024"]
        SX = score[model]["1024"]

        natural_direction = unit(
            word_vector(NW, "irregular") -
            word_vector(NW, "regular")
        )

        a = score_concept_name_vector(
            SX,
            rhythmic_name
        )
        b = score_concept_name_vector(
            SX,
            arhythmic_name
        )

        eeg_direction = unit(b - a)

        rows.append({
            "model": model,
            "regular→irregular vs EEG direction": float(
                natural_direction @ eeg_direction
            )
        })

    display(pd.DataFrame(rows).round(4))
else:
    print("Rhythmic/Arhythmic concept-name pair not found.")

## 9. Agreement of ordinary-word geometry across models

In [ ]:
vals = {}

for model in MODEL_FILES:
    X = natural[model]["1024"]
    S = X @ X.T

    mask = np.triu(
        np.ones_like(S, dtype=bool),
        k=1
    )

    vals[model] = S[mask]

corr = pd.DataFrame(
    index=MODEL_FILES,
    columns=MODEL_FILES,
    dtype=float,
)

for a in MODEL_FILES:
    for b in MODEL_FILES:
        corr.loc[a, b] = spearmanr(
            vals[a],
            vals[b]
        ).statistic

display(corr.round(3))

## 10. SCORE definitions versus definitions from other domains

The external control set contains eight deliberately varied domains:

- cardiology,
- cardiac surgery,
- orthopedics,
- dermatology,
- competitive rowing,
- volleyball,
- contract bridge,
- association football.

The definitions are concise paraphrases of authoritative web sources stored in `external_domain_definitions.csv`.

The goal is not to classify domains formally. We use them as controls to ask whether the embedding space distinguishes:

1. EEG terminology from other medical language,
2. medical language from a non-medical technical domain,
3. domain membership despite similar definition-like sentence structure.

In [ ]:
display(
    external_meta[
        external_meta["type"].eq("definition")
    ][
        ["domain", "concept", "text", "source_name"]
    ].reset_index(drop=True)
)

### Mean cosine similarity within and between domains

We compare definition embeddings only.

`SCORE` contains all SCORE definitions. Each external control domain contains ten concept-definition pairs.

A useful qualitative pattern would be:
- relatively high within-domain similarity,
- cardiology/cardiac-surgery closer to one another than either is to rowing,
- SCORE separated from rowing but potentially partly overlapping with cardiovascular medicine because all are biomedical English.

In [ ]:
def definition_domain_sets(model, space="1024"):
    Xs = score[model][space]
    Xe = external[model][space]

    score_idx = score_meta.index[
        score_meta["type"].eq("definition")
    ].to_numpy()

    ext_idx = external_meta.index[
        external_meta["type"].eq("definition")
    ].to_numpy()

    groups = {
        "SCORE": Xs[score_idx]
    }

    for domain in sorted(
        external_meta.loc[ext_idx, "domain"].unique()
    ):
        idx = external_meta.index[
            external_meta["type"].eq("definition") &
            external_meta["domain"].eq(domain)
        ].to_numpy()

        groups[domain] = Xe[idx]

    return groups


def mean_cross_cosine(A, B, same_group=False):
    S = A @ B.T

    if same_group:
        mask = ~np.eye(len(A), dtype=bool)
        return float(S[mask].mean())

    return float(S.mean())


for model in MODEL_FILES:
    groups = definition_domain_sets(model, "1024")
    names = list(groups)

    M = pd.DataFrame(
        index=names,
        columns=names,
        dtype=float
    )

    for a in names:
        for b in names:
            M.loc[a, b] = mean_cross_cosine(
                groups[a],
                groups[b],
                same_group=(a == b)
            )

    print("\nMODEL:", model)
    display(M.round(3))

### Domain centroids

Each domain centroid is the normalized mean of its definition vectors.

This is only a coarse summary, but it makes the overall domain arrangement easy to compare between model sizes.

In [ ]:
for model in MODEL_FILES:
    groups = definition_domain_sets(model, "1024")

    names = list(groups)
    C = []

    for name in names:
        c = groups[name].mean(axis=0)
        c = c / np.linalg.norm(c)
        C.append(c)

    C = np.vstack(C)

    print("\nMODEL:", model)
    display(
        pd.DataFrame(
            C @ C.T,
            index=names,
            columns=names
        ).round(3)
    )

### How close is each external definition to SCORE?

For every external definition, find the nearest SCORE definition.

This is useful for detecting accidental cross-domain semantic neighbours and for seeing whether larger models make domain-specific distinctions more or less pronounced.

In [ ]:
score_def_idx = score_meta.index[
    score_meta["type"].eq("definition")
].to_numpy()

score_def_names = score_meta.loc[
    score_def_idx,
    "concept"
].to_numpy()

external_def_idx = external_meta.index[
    external_meta["type"].eq("definition")
].to_numpy()

external_def_info = external_meta.loc[
    external_def_idx,
    ["domain", "concept"]
].reset_index(drop=True)


for model in MODEL_FILES:
    A = external[model]["1024"][external_def_idx]
    B = score[model]["1024"][score_def_idx]

    S = A @ B.T
    nn = np.argmax(S, axis=1)

    out = external_def_info.copy()
    out["nearest_SCORE_concept"] = score_def_names[nn]
    out["cosine"] = S[
        np.arange(len(A)),
        nn
    ]

    print("\nMODEL:", model)
    display(
        out.sort_values(
            ["domain", "cosine"],
            ascending=[True, False]
        ).round(3)
    )

### Concept-label → definition similarity across domains

This gives a useful control for the SCORE `concept → definition` analysis.

The external domains contain one short concept label and one definition per item, so we can ask whether specialized EEG labels are intrinsically harder for the model to connect to their definitions than labels from cardiology, cardiac surgery, or rowing.

Because the external sets are small, focus primarily on the cosine distributions rather than comparing top-k retrieval rates directly.

In [ ]:
def external_label_definition_cosines(X, meta):
    rows = []

    for row_id, g in meta.groupby("external_row_id", sort=True):
        c_idx = g.index[g["type"].eq("concept")][0]
        d_idx = g.index[g["type"].eq("definition")][0]

        rows.append({
            "domain": g["domain"].iloc[0],
            "concept": g["concept"].iloc[0],
            "cosine": float(X[c_idx] @ X[d_idx])
        })

    return pd.DataFrame(rows)


score_c_mask = score_meta["type"].eq("concept").to_numpy()
score_d_mask = score_meta["type"].eq("definition").to_numpy()

for model in MODEL_FILES:
    Xs = score[model]["1024"]

    score_cd = np.sum(
        Xs[score_c_mask] * Xs[score_d_mask],
        axis=1
    )

    ext_cd = external_label_definition_cosines(
        external[model]["1024"],
        external_meta
    )

    summary = [
        {
            "domain": "SCORE",
            "mean": score_cd.mean(),
            "median": np.median(score_cd),
            "min": score_cd.min(),
            "max": score_cd.max(),
            "n": len(score_cd),
        }
    ]

    for domain, g in ext_cd.groupby("domain"):
        summary.append({
            "domain": domain,
            "mean": g["cosine"].mean(),
            "median": g["cosine"].median(),
            "min": g["cosine"].min(),
            "max": g["cosine"].max(),
            "n": len(g),
        })

    print("\nMODEL:", model)
    display(pd.DataFrame(summary).round(3))

### PCA of definition embeddings from all domains

PCA is fitted separately for each model on SCORE + external definition vectors at 1024 dimensions.

This plot is descriptive only. Two PCA components cannot be treated as the full embedding geometry.

In [ ]:
for model in MODEL_FILES:
    Xs = score[model]["1024"]
    Xe = external[model]["1024"]

    score_idx = score_meta.index[
        score_meta["type"].eq("definition")
    ].to_numpy()

    ext_idx = external_meta.index[
        external_meta["type"].eq("definition")
    ].to_numpy()

    X_plot = np.vstack([
        Xs[score_idx],
        Xe[ext_idx]
    ])

    labels = np.concatenate([
        np.repeat("SCORE", len(score_idx)),
        external_meta.loc[
            ext_idx,
            "domain"
        ].to_numpy()
    ])

    pca = PCA(n_components=2)
    Z = pca.fit_transform(X_plot)

    plt.figure(figsize=(9, 7))

    for domain in pd.unique(labels):
        mask = labels == domain
        plt.scatter(
            Z[mask, 0],
            Z[mask, 1],
            label=domain,
            alpha=0.7
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(
        f"{model} — SCORE vs external definitions\n"
        f"variance explained = "
        f"{pca.explained_variance_ratio_.sum():.3f}"
    )
    plt.legend()
    plt.show()

## 11. Preliminary questions for interpretation

Use the outputs above to ask:

- Does model scaling improve concept → definition retrieval at matched 1024 dimensions?
- Does it also improve at native dimensions, or is some gain attributable to dimensionality?
- Are paraphrase similarities already saturated at 0.6B?
- Do difficult EEG contrasts move apart, remain almost identical, or behave inconsistently?
- Is ordinary-language relational geometry more regular in larger models?
- Do all three models broadly agree on the ordering of pairwise SCORE similarities?
- Is SCORE more similar to other medical definitions than to rowing definitions?
- Are concept-label → definition similarities unusually weak for EEG compared with the control domains?

These are exploratory diagnostics, not final statistical tests.